# MUFASA entity-resolution evaluation

Thin local/Kaggle control and inspection surface for `scripts.entity_resolution`. Matching, validation and registry rules live only in the reusable module. `COMMIT` is deliberately `False` by default.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

# Set these explicitly on Kaggle after attaching the repository, extraction
# output and corpus output datasets. /kaggle/input is read-only; output must
# remain below /kaggle/working to persist when you Save Version.
MUFASA_ROOT = Path.cwd().parent if Path.cwd().name == '03-retrieval' else Path.cwd() / 'MUFASA'
EXTRACTION_DIR = None
DOCUMENTS_PATH = None
REGISTRY_DIR = None              # None means the empty version-0 registry
AUTHORITY_DIR = None             # None means no authority lane
AUTHORITY_HINTS_PATH = None
POLICY_PATH = MUFASA_ROOT / 'scripts/entity_resolution/policies/mufasa-v1.yaml'
OUTPUT_ROOT = Path('/kaggle/working/mufasa_entity_resolution') if Path('/kaggle/working').is_dir() else MUFASA_ROOT / 'artifacts/entity_resolution'
WORKERS = max(1, min(8, os.cpu_count() or 1))
COMMIT = False
APPROVED_PROPOSAL_IDS = []
MENTION_GOLD_PATH = None
PAIR_GOLD_PATH = None

print({'MUFASA_ROOT': str(MUFASA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'WORKERS': WORKERS, 'COMMIT': COMMIT})

In [ ]:
# Dependency installation is never attempted silently. If this cell reports
# a missing package, run the following explicit command, restart, then rerun:
# %pip install -r /path/to/MUFASA/scripts/entity_resolution/requirements.txt
if not (MUFASA_ROOT / 'scripts/entity_resolution').is_dir():
    raise FileNotFoundError(f'MUFASA_ROOT is wrong: {MUFASA_ROOT}')
sys.path.insert(0, str(MUFASA_ROOT))
from scripts.entity_resolution import (
    commit_resolution_run, evaluate_run, load_authority_snapshot,
    load_mufasa_inputs, load_policy, load_registry_snapshot, load_resolution_run, resolve_batch,
    write_registry_snapshot, write_resolution_run,
)
from scripts.entity_resolution.audit import build_review_rows, registry_diff, run_summary
print('resolver package imported from', MUFASA_ROOT)

In [ ]:
if EXTRACTION_DIR is None or DOCUMENTS_PATH is None:
    raise ValueError('Set EXTRACTION_DIR and DOCUMENTS_PATH in the controls cell.')
EXTRACTION_DIR = Path(EXTRACTION_DIR)
DOCUMENTS_PATH = Path(DOCUMENTS_PATH)
for required in (EXTRACTION_DIR / 'current-generation.json', EXTRACTION_DIR / 'manifest.json', EXTRACTION_DIR / 'run_summary.json', DOCUMENTS_PATH):
    if not required.exists():
        raise FileNotFoundError(required)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
policy = load_policy(POLICY_PATH)
authorities = load_authority_snapshot(AUTHORITY_DIR)
registry = load_registry_snapshot(REGISTRY_DIR, authority_snapshot=authorities if authorities.records else None)
inputs = load_mufasa_inputs(EXTRACTION_DIR, DOCUMENTS_PATH, policy, authority_hints_path=AUTHORITY_HINTS_PATH)
print({'valid_mentions': len(inputs.mentions), 'adapter_invalid': len(inputs.invalid_mentions), 'registry_version': registry.version, 'authority_records': len(authorities.records), 'input_fingerprint': inputs.input_fingerprint})

In [ ]:
# Mutation-free dry run. Embedding recall is intentionally absent unless a
# pinned real backend is explicitly passed; lexical scores never substitute.
execution = resolve_batch(
    inputs.mentions, registry, policy, authority_snapshot=authorities,
    authority_hints=inputs.authority_hints, invalid_mentions=inputs.invalid_mentions,
    input_fingerprint=inputs.input_fingerprint, workers=WORKERS,
)
run_capabilities = dict(execution.capability_manifest)
run_capabilities['extraction_input'] = {
    'generation_id': inputs.extraction_generation_id,
    'source_fingerprint': inputs.source_fingerprint,
    'settings_hash': inputs.settings_hash,
    'schema_version': inputs.extraction_schema_version,
    'prompt_version': inputs.prompt_version,
}
summary = run_summary(execution.run)
display(pd.DataFrame([summary]))
print(json.dumps(execution.capability_manifest, indent=2, sort_keys=True))

In [ ]:
decisions = pd.DataFrame([{
    'mention_id': d.mention_id, 'status': d.status.value, 'method': d.method.value,
    'concept_id': d.concept_id, 'instance_id': d.instance_id,
    'proposal_id': d.proposal_id, 'reasons': ', '.join(d.reason_codes),
} for d in execution.run.decisions])
display(decisions['status'].value_counts(dropna=False).rename_axis('status').to_frame('mentions'))
display(decisions['method'].value_counts(dropna=False).rename_axis('method').to_frame('mentions'))
review = pd.DataFrame(build_review_rows(execution.run))
display(review.head(30))
if not review.empty:
    review.groupby('entity_type')['occurrence_count'].sum().sort_values().plot.barh(title='Review occurrences by entity type', figsize=(8, 5));

In [ ]:
dry_run_dir = OUTPUT_ROOT / execution.run.run_id / 'dry-run'
write_resolution_run(execution.run, dry_run_dir, conflicts=execution.conflicts, capability_manifest=run_capabilities)
published_dry_run = load_resolution_run(dry_run_dir)
pointer = json.loads((dry_run_dir / 'current-run.json').read_text(encoding='utf-8'))
generation_dir = dry_run_dir / pointer['directory']
print('dry-run generation:', generation_dir, 'verified run:', published_dry_run.run_id)
for path in sorted(generation_dir.iterdir()): print(' ', path.name, path.stat().st_size)

In [ ]:
# Explicit publication gate. Review the dry-run tables first, then change
# COMMIT in the controls cell and rerun from the top.
if COMMIT:
    committed = commit_resolution_run(
        execution, registry, policy, authority_snapshot=authorities,
        approved_proposal_ids=APPROVED_PROPOSAL_IDS,
    )
    committed_dir = OUTPUT_ROOT / execution.run.run_id / 'committed-run'
    registry_dir = OUTPUT_ROOT / 'registries' / committed.registry.version
    write_resolution_run(committed.run, committed_dir, conflicts=execution.conflicts, capability_manifest=run_capabilities, registry_diff_value=committed.diff)
    write_registry_snapshot(committed.registry, registry_dir, authority_snapshot=authorities if authorities.records else None, manifest_extra={'source_run_id': execution.run.run_id, 'policy_version': policy.version, 'policy_hash': policy.content_hash})
    print('committed registry:', registry_dir)
else:
    print('COMMIT=False: registry was not mutated or published.')

In [ ]:
def read_gold(path):
    if path is None: return None
    path = Path(path)
    return pd.read_parquet(path) if path.suffix.lower() == '.parquet' else pd.read_csv(path)
if MENTION_GOLD_PATH or PAIR_GOLD_PATH:
    # Preview auto-approved proposals in memory so gold sees their stable IDs.
    # This does not publish or mutate a registry; COMMIT remains the write gate.
    if COMMIT:
        evaluation_run = committed.run
        evaluation_mode = 'committed'
    else:
        preview = commit_resolution_run(execution, registry, policy, authority_snapshot=authorities)
        evaluation_run = preview.run
        evaluation_mode = 'in-memory auto-proposal preview'
    print('evaluation mode:', evaluation_mode)
    report = evaluate_run(evaluation_run, mention_gold=read_gold(MENTION_GOLD_PATH), pair_gold=read_gold(PAIR_GOLD_PATH))
    display(pd.DataFrame([report.metrics]))
    display(pd.DataFrame(report.by_type))
    display(pd.DataFrame(report.errors).head(50))
    print('release warnings:', report.release_warnings)
else:
    print('No resolution gold paths configured; no accuracy claim was computed.')